# Two combinations, no training: a combined AD-vs-FTD score and an ensemble of five graphs

Pre-registered in `docs/PREREG_2026-11-16.md` (addendum, written before either was computed). This reads saved
out-of-fold outputs on repeats 1–3, where every input shares folds, and uses the same 2000 subject bootstraps as
`Final_scoreboard_ds004504.ipynb`.

1. **A combined AD-vs-FTD score** from three weak scores that barely agree with each other: the 3-class GAT, C3 on
   ds006036 and the aperiodic exponent. In each fold every score is standardised on the training patients, then
   the three are averaged with equal weights. It replaces C3 as stage 2 only if its gain over C3 has a lower 95 %
   bound above 0.
2. **(2a) An ensemble of the five 3-class GATs** (PSD, Pearson, PLV, AEC-c and Granger graphs). It is reported
   beside the final model and does not replace it.

Writes `results_final/combine/`. Under a minute.

In [1]:
# ===========================================================================
# CELL 1 — INPUTS (the same loaders as Final_scoreboard_ds004504.ipynb)
# ===========================================================================
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm

WORK_DIR = Path.cwd()
EXP_DIR  = WORK_DIR / "experiments_ad_vs_ftd" / "results"
OUT_DIR  = WORK_DIR / "results_final" / "combine"
OUT_DIR.mkdir(parents=True, exist_ok=True)
HC, AD, FTD = 0, 1, 2
SEED, N_BOOT, R = 42, 2000, 3
GAT_DIRS = ["results_gat_multiclass"] + [f"results_gat_multiclass_{g}" for g in ("pearson", "plv", "aec", "granger")]

_n   = np.array([len(np.load(WORK_DIR / "subject_cache" / f"sub-{i:03d}.npz")["y"]) for i in range(1, 89)])
SIDX = np.repeat(np.arange(88), _n)
Y    = np.array([AD if i <= 36 else HC if i <= 65 else FTD for i in range(1, 89)])
SUBJ = [f"sub-{i:03d}" for i in range(1, 89)]


def subj_of(o):
    """(repeats, windows, K) -> (3, 88, K): mean over each subject's windows."""
    o = o[:R]
    assert o.shape[1] == len(SIDX) and np.isfinite(o).all()
    return np.stack([np.stack([np.bincount(SIDX, o[r][:, k]) / _n for k in range(o.shape[2])], -1) for r in range(R)])


GAT_W  = {d: np.load(WORK_DIR / d / "oof_prob.npy") for d in GAT_DIRS}
VOTE_W = {d: np.load(WORK_DIR / d / "ensemble_oof_prob.npy") for d in GAT_DIRS}
for d in GAT_DIRS:            # every twin's soft vote uses the same band-power floor (same folds, deterministic fit)
    assert np.allclose(2 * VOTE_W[d][:R] - GAT_W[d][:R], 2 * VOTE_W[GAT_DIRS[0]][:R] - GAT_W[GAT_DIRS[0]][:R], atol=1e-8)

_s2  = np.load(EXP_DIR / "riemann" / "stage2_oof.npz", allow_pickle=True)
_e10 = np.load(EXP_DIR / "c3_ds006036" / "c3_oof.npz", allow_pickle=True)
assert (_s2["labels"] == Y).all() and (_e10["fold_of"] == _s2["fold_of"]).all()
FOLD_OF = _s2["fold_of"][:R]                                              # the 3-class notebook's folds, repeats 1-3
C3 = _e10["q_ad"][list(_e10["runs"]).index("ds006036, 44 s (primary)")][:R]   # (3, 88)
EXPO = (pd.read_csv(EXP_DIR / "aperiodic" / "subject_scores.csv").set_index("participant_id")
          .loc[SUBJ, "exponent_2s"].to_numpy(float))                     # (88,), higher = AD (Experiment 8)
STAGE1 = subj_of(np.load(WORK_DIR / "results_gat_dementia" / "ensemble_oof_prob.npy"))[..., 1]   # (3, 88) P(dementia)
print(f"{len(GAT_DIRS)} GATs, C3 {C3.shape}, exponent {EXPO.shape}, stage 1 {STAGE1.shape}; floors identical across twins")


5 GATs, C3 (3, 88), exponent (88,), stage 1 (3, 88); floors identical across twins


In [2]:
# ===========================================================================
# CELL 2 — METRICS AND THE SHARED BOOTSTRAP (as Final_scoreboard_ds004504.ipynb)
# ===========================================================================
_rng = np.random.default_rng(SEED)
_cls = {c: np.flatnonzero(Y == c) for c in (HC, AD, FTD)}
BOOT = np.stack([np.concatenate([_rng.choice(i, len(i)) for i in _cls.values()]) for _ in range(N_BOOT)])
FULL = np.concatenate(list(_cls.values()))


def _auc(pos, neg):
    d = pos[:, None] - neg[None, :]
    return (d > 0).mean() + 0.5 * (d == 0).mean()


def three(P, idx):
    y, out = Y[idx], []
    for p in P[:, idx]:
        pred = p.argmax(1)
        cm = np.bincount(y * 3 + pred, minlength=9).reshape(3, 3)
        rec = cm.diagonal() / cm.sum(1)
        prec = np.divide(cm.diagonal(), cm.sum(0), out=np.zeros(3), where=cm.sum(0) > 0)
        f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(3), where=(prec + rec) > 0)
        s = p[:, AD] / (p[:, AD] + p[:, FTD])
        out.append(dict(acc=(pred == y).mean(), macro_f1=f1.mean(), macro_recall=rec.mean(),
                        recall_HC=rec[HC], recall_AD=rec[AD], recall_FTD=rec[FTD],
                        auc_AD_FTD=_auc(s[y == AD], s[y == FTD])))
    return pd.DataFrame(out).mean()


def ad_ftd(S, idx):
    """S (3, 88) scores, higher = AD. AD-vs-FTD AUC averaged over repeats."""
    y = Y[idx]
    return pd.Series(dict(auc=np.mean([_auc(s[y == AD], s[y == FTD]) for s in S[:, idx]])))


def evaluate(fn, X):
    return fn(X, FULL), pd.DataFrame([fn(X, b) for b in BOOT])


def paired(da, db, k):
    d = da[k] - db[k]
    return dict(lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5),
                p=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))


ci  = lambda dist, k: np.percentile(dist[k], [2.5, 97.5])
fmt = lambda v, d, k: f"{v[k]:.3f} [{ci(d, k)[0]:.3f}, {ci(d, k)[1]:.3f}]"


In [3]:
# ===========================================================================
# CELL 3 — 1. THE COMBINED AD-VS-FTD SCORE (fold-wise standardisation on training patients)
# ===========================================================================
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
_gat3 = subj_of(GAT_W["results_gat_multiclass"])
COMP = {"3-class GAT": logit(_gat3[..., AD] / (_gat3[..., AD] + _gat3[..., FTD])),     # (3, 88)
        "C3, ds006036": logit(C3),
        "aperiodic exponent": np.tile(EXPO, (R, 1))}
COMB = np.full((R, 88), np.nan)
for r in range(R):
    for f in np.unique(FOLD_OF[r]):
        test = np.flatnonzero(FOLD_OF[r] == f)
        train = np.flatnonzero((FOLD_OF[r] != f) & (Y != HC))              # the fold's training dementia patients
        z = lambda v, idx: (v[r, idx] - v[r, train].mean()) / v[r, train].std(ddof=1)
        avg_test = np.mean([z(v, test) for v in COMP.values()], 0)
        avg_train = np.mean([z(v, train) for v in COMP.values()], 0)
        COMB[r, test] = norm.cdf(avg_test / avg_train.std(ddof=1))
assert np.isfinite(COMB).all()

SCORES = {**{n: v for n, v in COMP.items()}, "combined (equal-weight average)": COMB}
EVS = {n: evaluate(ad_ftd, S) for n, S in SCORES.items()}
print("AD vs FTD, 59 patients, mean of repeats 1-3 [95 % CI]\n")
for n, (v, d) in EVS.items():
    print(f"  {n:34s} {fmt(v, d, 'auc')}")
GAIN = paired(EVS["combined (equal-weight average)"][1], EVS["C3, ds006036"][1], "auc")
GAIN["diff"] = EVS["combined (equal-weight average)"][0]["auc"] - EVS["C3, ds006036"][0]["auc"]
ADOPT = bool(GAIN["lo"] > 0)
print(f"\ncombined minus C3 alone: {GAIN['diff']:+.3f} [{GAIN['lo']:+.3f}, {GAIN['hi']:+.3f}], p {GAIN['p']:.3f}")
print(f"adopted as stage 2 (pre-registered rule: lower bound > 0): {ADOPT}")


AD vs FTD, 59 patients, mean of repeats 1-3 [95 % CI]

  3-class GAT                        0.595 [0.472, 0.714]
  C3, ds006036                       0.644 [0.511, 0.769]
  aperiodic exponent                 0.615 [0.452, 0.769]
  combined (equal-weight average)    0.641 [0.514, 0.772]

combined minus C3 alone: -0.003 [-0.119, +0.116], p 0.974
adopted as stage 2 (pre-registered rule: lower bound > 0): False


In [4]:
# ===========================================================================
# CELL 4 — THE COMBINED SCORE AS STAGE 2 OF THE 3-CLASS MODEL
# ===========================================================================
two = lambda q: np.stack([1 - STAGE1, STAGE1 * q, STAGE1 * (1 - q)], -1)
EV3 = {"final model: stage 2 = C3 (scoreboard)": evaluate(three, two(C3)),
       "stage 2 = combined score": evaluate(three, two(COMB)),
       "soft vote, PSD graphs (today)": evaluate(three, subj_of(VOTE_W["results_gat_multiclass"]))}
KEYS = ["acc", "macro_f1", "macro_recall", "recall_HC", "recall_AD", "recall_FTD", "auc_AD_FTD"]
T3 = pd.DataFrame({n: {k: fmt(v, d, k) for k in KEYS} for n, (v, d) in EV3.items()}).T
pd.set_option("display.width", 250, "display.max_colwidth", 30)
print("3-class, stage 1 = the dementia soft vote, mean of repeats 1-3 [95 % CI]\n")
print(T3.to_string())
_t = paired(EV3["stage 2 = combined score"][1], EV3["final model: stage 2 = C3 (scoreboard)"][1], "macro_recall")
print(f"\ncombined minus C3 as stage 2, macro recall: "
      f"{EV3['stage 2 = combined score'][0]['macro_recall'] - EV3['final model: stage 2 = C3 (scoreboard)'][0]['macro_recall']:+.3f} "
      f"[{_t['lo']:+.3f}, {_t['hi']:+.3f}], p {_t['p']:.3f}")


3-class, stage 1 = the dementia soft vote, mean of repeats 1-3 [95 % CI]

                                                         acc              macro_f1          macro_recall             recall_HC             recall_AD            recall_FTD            auc_AD_FTD
final model: stage 2 = C3 (scoreboard)  0.606 [0.515, 0.693]  0.589 [0.497, 0.673]  0.597 [0.505, 0.682]  0.782 [0.632, 0.908]  0.574 [0.426, 0.713]  0.435 [0.261, 0.609]  0.644 [0.511, 0.769]
stage 2 = combined score                0.587 [0.500, 0.670]  0.568 [0.482, 0.649]  0.577 [0.492, 0.658]  0.793 [0.644, 0.920]  0.546 [0.407, 0.694]  0.391 [0.232, 0.536]  0.641 [0.514, 0.772]
soft vote, PSD graphs (today)           0.625 [0.538, 0.705]  0.592 [0.502, 0.680]  0.600 [0.512, 0.684]  0.782 [0.632, 0.908]  0.685 [0.556, 0.806]  0.333 [0.174, 0.493]  0.601 [0.464, 0.735]

combined minus C3 as stage 2, macro recall: -0.020 [-0.081, +0.044], p 0.533


In [5]:
# ===========================================================================
# CELL 5 — 2a. AN ENSEMBLE OF THE FIVE 3-CLASS GATS
# ===========================================================================
ENS_GAT  = subj_of(np.mean([GAT_W[d] for d in GAT_DIRS], 0))
ENS_VOTE = subj_of(np.mean([VOTE_W[d] for d in GAT_DIRS], 0))      # = (mean of the five GATs + band power) / 2
EVE = {"GAT, PSD graphs (today)": evaluate(three, subj_of(GAT_W["results_gat_multiclass"])),
       "ensemble of five GATs": evaluate(three, ENS_GAT),
       "soft vote, PSD graphs (today)": EV3["soft vote, PSD graphs (today)"],
       "ensemble of five GATs + band power": evaluate(three, ENS_VOTE)}
TE = pd.DataFrame({n: {k: fmt(v, d, k) for k in KEYS} for n, (v, d) in EVE.items()}).T
print("3-class, subject level, mean of repeats 1-3 [95 % CI]\n")
print(TE.to_string())
rows = []
for k in ("macro_recall", "auc_AD_FTD"):
    pair = [("ensemble of five GATs", "GAT, PSD graphs (today)"),
            ("ensemble of five GATs + band power", "soft vote, PSD graphs (today)")]
    ps = [paired(EVE[a][1], EVE[b][1], k) for a, b in pair]
    order = np.argsort([p["p"] for p in ps]); run = 0.0; holm = [0.0, 0.0]
    for j, i in enumerate(order):
        run = max(run, min(1.0, (2 - j) * ps[i]["p"])); holm[i] = run
    rows += [dict(metric=k, comparison=f"{a} - {b}", diff=EVE[a][0][k] - EVE[b][0][k], **p, p_holm=h)
             for (a, b), p, h in zip(pair, ps, holm)]
TESTS_E = pd.DataFrame(rows)
print("\npaired bootstrap; Holm over the two comparisons, per metric\n")
print(TESTS_E.round(3).to_string(index=False))


3-class, subject level, mean of repeats 1-3 [95 % CI]

                                                     acc              macro_f1          macro_recall             recall_HC             recall_AD            recall_FTD            auc_AD_FTD
GAT, PSD graphs (today)             0.621 [0.545, 0.693]  0.565 [0.487, 0.642]  0.583 [0.509, 0.656]  0.782 [0.632, 0.908]  0.750 [0.630, 0.861]  0.217 [0.101, 0.348]  0.595 [0.472, 0.714]
ensemble of five GATs               0.629 [0.557, 0.701]  0.576 [0.501, 0.646]  0.595 [0.526, 0.663]  0.839 [0.713, 0.954]  0.713 [0.583, 0.833]  0.232 [0.130, 0.348]  0.597 [0.463, 0.735]
soft vote, PSD graphs (today)       0.625 [0.538, 0.705]  0.592 [0.502, 0.680]  0.600 [0.512, 0.684]  0.782 [0.632, 0.908]  0.685 [0.556, 0.806]  0.333 [0.174, 0.493]  0.601 [0.464, 0.735]
ensemble of five GATs + band power  0.621 [0.530, 0.705]  0.592 [0.501, 0.681]  0.600 [0.512, 0.686]  0.805 [0.667, 0.920]  0.648 [0.509, 0.778]  0.348 [0.188, 0.522]  0.598 [0.459, 0.729]


In [6]:
# ===========================================================================
# CELL 6 — SAVE
# ===========================================================================
pd.DataFrame({n: {"auc": v["auc"], "lo": ci(d, "auc")[0], "hi": ci(d, "auc")[1]} for n, (v, d) in EVS.items()}).T \
  .to_csv(OUT_DIR / "ad_ftd_scores.csv")
T3.to_csv(OUT_DIR / "three_class_stage2.csv")
TE.to_csv(OUT_DIR / "graph_ensemble.csv")
TESTS_E.to_csv(OUT_DIR / "graph_ensemble_tests.csv", index=False)
np.savez_compressed(OUT_DIR / "combined_q.npz", q=COMB, fold_of=FOLD_OF, y=Y)
(OUT_DIR / "summary.json").write_text(json.dumps(dict(
    combined_auc=EVS["combined (equal-weight average)"][0]["auc"],
    combined_ci=list(ci(EVS["combined (equal-weight average)"][1], "auc")),
    gain_vs_c3={k: float(v) for k, v in GAIN.items()}, adopted_as_stage_2=ADOPT,
    graph_ensemble_tests=TESTS_E.round(4).to_dict("records")), indent=1, default=float), encoding="utf-8")
print(f"saved to {OUT_DIR}:", ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))


saved to C:\Projects\Major Project\results_final\combine: ad_ftd_scores.csv, combined_q.npz, graph_ensemble.csv, graph_ensemble_tests.csv, summary.json, three_class_stage2.csv
